# Symmetry of the Y₂Ir₂O₇ models (paper `tab:symmetry-yio`)

Do the Wannier models behind ∂θ/∂Q obey the magnetic point group of their structure? Production runs: self-test base
214249 (series windows 12.422 / 22.644 eV, conv_thr 1e-10) and the Q = 0.01 Å Γ₂⁻ runs 214147 / 214151 / 214155. The
models are built as in production (H and A(R) from the `.chk`, folded on the base centres' nearest atoms). Functions:
`axion_phonon/symmetry.py`; the group theory is in this folder (`symm.pdf`, `../../yiro-mbt-symm.txt`).

**The groups** are read from each structure's pw.x output (QE's Cartesian matrices and 'Time Reversal' flags):
all-in-all-out base m-3m′ (48 operations, 24 primed; MSG $Fd\bar3m'$), each Γ₂⁻ displaced structure −4′3m′ (24, 12
primed; MSG $F\bar4'3m'$). QE lists the whole group (no supercell), so nothing is added.

**Three tests per model:**
1. DFT eigenvalues on the 8³ Wannier mesh, lowest 260 bands: e_n(g k) = e_n(k) (k → −g k for a primed g). Needs
   only the `.eig` and `.win`, so it runs here for every structure.
2. Interpolated bands at the test k points.
3. Curvature. AIAO breaks time reversal, so the Berry curvature of the 156 occupied bands is not zero; the test is
   its band trace, the Cartesian pseudovector ω_a = Re Tr F_a (F_a = ε_abc Ω_bc/2): ω(g k) = det(R) R ω(k), and
   ω(−R k) = −det(R) R ω(k) for a primed g, separately for each part of eq (24) and for their sum.

**Controls**, which must fail: the fourfold operations C₄z and S₄z with whichever priming is not in the group. Plain
inversion is not used for the displaced structures: Q = 0.01 Å breaks it only at O(Q), below the residual floor.

**Which models run here:** the base `.chk` is in `data/`. The displaced structures' `.chk` are on the cluster only:
their test 1 runs here, tests 2-3 there (results loaded when copied to `data/`).

In [1]:
# ---- settings
from pathlib import Path
DATA = Path("/Users/treycole/Repos/axion-phonon/data/Y2Ir2O7")
PHONON = DATA / "phonon/GM2-"
RUNS = {   # structure: (Wannier run folder, pw.x scf.out with its group)
    "base": (DATA / "base/lda/w90_trial02_selftest_ampwin10_214249", DATA / "base/lda/scf_conv1e-10_213998/Y2Ir2O7.scf.out"),
    "mode1": (PHONON / "mode1/Q_0.01A/lda/w90_trial02_ampwin10_2026-10-18_214147",
              PHONON / "mode1/Q_0.01A/lda/scf_nscf_conv1e-10_213999/Y2Ir2O7.scf.out"),
    "mode2": (PHONON / "mode2/Q_0.01A/lda/w90_trial02_ampwin10_2026-10-18_214151",
              PHONON / "mode2/Q_0.01A/lda/scf_nscf_conv1e-10_214003/Y2Ir2O7.scf.out"),
    "mode3": (PHONON / "mode3/Q_0.01A/lda/w90_trial02_ampwin10_2026-10-18_214155",
              PHONON / "mode3/Q_0.01A/lda/scf_nscf_conv1e-10_214016/Y2Ir2O7.scf.out"),
}
CLUSTER = DATA / "symmetry"          # cluster results of the displaced structures: CLUSTER/<structure>.npz
PREFIX = "Y2Ir2O7"
N_OCC = 156
EIG_BANDS = 260                      # lowest bands in test 1 (paper)
N_GENERIC = 7                        # random k: Cartesian components uniform in +-GENERIC_HALF_WIDTH
GENERIC_HALF_WIDTH = 0.23            # 1/A
SEED = 20260825                      # as the 10-26 table: the same k
THRESHOLDS = {"dft_ueV": 2.0, "bands_meV": 1e-3, "internal": 1e-5, "cross": 1e-3, "external": 1e-3, "total": 1e-4,
              "control_min": 0.5}    # maxima over the operations other than the identity; each control at least
SAVE = DATA / "symmetry"             # summary.csv and one npz per model (read by tests/regression); None to skip
# ---------------


In [2]:
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "/Users/treycole/Repos/axion-phonon")
from axion_phonon import symmetry, wannier
from axion_phonon.io import wannier90

pd.set_option("display.width", 200, "display.max_columns", 20, "display.max_colwidth", 60)
C4Z = np.array([[0.0, -1, 0], [1, 0, 0], [0, 0, 1]])
S4Z = np.array([[0.0, 1, 0], [-1, 0, 0], [0, 0, -1]])
CONTROLS = (("C4z unprimed", C4Z, False), ("C4z T", C4Z, True), ("S4z unprimed", S4Z, False), ("S4z T", S4Z, True))


def mesh_run(folder):
    """What test 1 needs from a Wannier run without its .chk: the mesh (.win kpoints), the .eig and the cell."""
    win = wannier90.read_win(folder / f"{PREFIX}.win")
    text = (folder / f"{PREFIX}.win").read_text().lower()
    block = text.split("begin kpoints")[1].split("end kpoints")[0]
    kpts = np.array([l.split()[:3] for l in block.strip().splitlines()], float)
    return {"kpts": kpts, "energies": wannier90.read_eig(folder / f"{PREFIX}.eig", len(kpts)),
            "reciprocal": 2 * np.pi * np.linalg.inv(win["lattice"]).T, "mp_grid": win["mp_grid"]}


def matrix_text(R):
    return " / ".join(" ".join(f"{x:+.3f}".replace("+0.000", " 0") for x in row) for row in R)


## The groups

In [3]:
groups = {s: symmetry.group(scf) for s, (_, scf) in RUNS.items()}
for s in ("mode2", "mode3"):
    assert np.allclose(groups[s][0], groups["mode1"][0]) and np.array_equal(groups[s][1], groups["mode1"][1]), s
for label, g in (("base", groups["base"]), ("displaced structures (modes 1-3)", groups["mode1"])):
    ops, primed, names, listed = g
    ctrl = symmetry.controls(ops, primed, CONTROLS)
    print(f"{label}: {len(ops)} operations, {int(primed.sum())} primed; controls: {[c[0] for c in ctrl]}")
    display(pd.DataFrame({"operation": names, "R (Cartesian, by rows)": [matrix_text(R) for R in ops],
                          "det": np.rint(np.linalg.det(ops)).astype(int), "primed": primed}))


base: 48 operations, 24 primed; controls: ['C4z unprimed', 'S4z unprimed']


,operation,"R (Cartesian, by rows)",det,primed
0,identity,+1.000 0 0 / 0 +1.000 0 / 0 0 +1.000,1,False
1,"180 deg rotation - cart. axis [0,0,1]",-1.000 0 0 / 0 -1.000 0 / 0 0 +1.000,1,False
2,"180 deg rotation - cart. axis [0,1,0]",-1.000 0 0 / 0 +1.000 0 / 0 0 -1.000,1,False
3,"180 deg rotation - cart. axis [1,0,0]",+1.000 0 0 / 0 -1.000 0 / 0 0 -1.000,1,False
4,"180 deg rotation - cart. axis [1,1,0] T",0 +1.000 0 / +1.000 0 0 / 0 0 -1.000,1,True
5,"180 deg rotation - cart. axis [1,-1,0] T",0 -1.000 0 / -1.000 0 0 / 0 0 -1.000,1,True
6,"90 deg rotation - cart. axis [0,0,-1] T",0 +1.000 0 / -1.000 0 0 / 0 0 +1.000,1,True
7,"90 deg rotation - cart. axis [0,0,1] T",0 -1.000 0 / +1.000 0 0 / 0 0 +1.000,1,True
8,"180 deg rotation - cart. axis [1,0,1] T",0 0 +1.000 / 0 -1.000 0 / +1.000 0 0,1,True
9,"180 deg rotation - cart. axis [-1,0,1] T",0 0 -1.000 / 0 -1.000 0 / -1.000 0 0,1,True


displaced structures (modes 1-3): 24 operations, 12 primed; controls: ['C4z unprimed', 'C4z T', 'S4z unprimed']


,operation,"R (Cartesian, by rows)",det,primed
0,identity,+1.000 0 0 / 0 +1.000 0 / 0 0 +1.000,1,False
1,"180 deg rotation - cart. axis [0,0,1]",-1.000 0 0 / 0 -1.000 0 / 0 0 +1.000,1,False
2,"180 deg rotation - cart. axis [0,1,0]",-1.000 0 0 / 0 +1.000 0 / 0 0 -1.000,1,False
3,"180 deg rotation - cart. axis [1,0,0]",+1.000 0 0 / 0 -1.000 0 / 0 0 -1.000,1,False
4,"120 deg rotation - cart. axis [-1,-1,-1]",0 +1.000 0 / 0 0 +1.000 / +1.000 0 0,1,False
5,"120 deg rotation - cart. axis [-1,1,1]",0 -1.000 0 / 0 0 +1.000 / -1.000 0 0,1,False
6,"120 deg rotation - cart. axis [1,1,-1]",0 +1.000 0 / 0 0 -1.000 / -1.000 0 0,1,False
7,"120 deg rotation - cart. axis [1,-1,1]",0 -1.000 0 / 0 0 -1.000 / +1.000 0 0,1,False
8,"120 deg rotation - cart. axis [1,1,1]",0 0 +1.000 / +1.000 0 0 / 0 +1.000 0,1,False
9,"120 deg rotation - cart. axis [-1,1,-1]",0 0 +1.000 / -1.000 0 0 / 0 -1.000 0,1,False


## The k points

Seven random points, the same as the 10-26 table: Cartesian components uniform in ±0.23 Å⁻¹ (the zone of the fcc
cell reaches 0.62 Å⁻¹ along [100], to X).

In [4]:
reciprocal = mesh_run(RUNS["base"][0])["reciprocal"]
k = symmetry.random_k(reciprocal, N_GENERIC, GENERIC_HALF_WIDTH, np.random.default_rng(SEED))
display(pd.DataFrame({"k reduced": list(np.round(k, 4)), "k Cartesian (1/A)": list(np.round(k @ reciprocal, 4)),
                      "|k| (1/A)": np.round(np.linalg.norm(k @ reciprocal, axis=1), 4)}))


,k reduced,k Cartesian (1/A),|k| (1/A)
0,"[-0.0265, 0.0678, 0.094]","[0.1161, -0.0002, -0.0325]",0.1206
1,"[-0.3171, -0.1631, -0.1917]","[-0.0233, -0.2132, -0.1779]",0.2786
2,"[0.2987, 0.2469, 0.2546]","[0.125, 0.1889, 0.1795]",0.2890
3,"[0.0366, -0.2611, 0.0434]","[-0.1568, 0.2103, -0.1652]",0.3101
4,"[0.0306, 0.0416, -0.0232]","[-0.0075, -0.0211, 0.0588]",0.0629
5,"[0.1583, 0.0079, -0.0346]","[-0.1141, 0.0714, 0.1238]",0.1829
6,"[-0.1646, 0.0465, -0.0053]","[0.1269, -0.1334, -0.0696]",0.1969


## Test 1: DFT eigenvalues on the mesh (lowest 260 bands)

In [5]:
dft = {}
for s, (folder, _) in RUNS.items():
    ops, primed, names, listed = groups[s]
    run = mesh_run(folder)
    dft[s] = symmetry.dft_residuals(run, ops, primed, EIG_BANDS)
    print(f"{s:6s}: {len(ops) - 1} operations besides the identity: max {dft[s][1:, 0].max():.2f} ueV "
          f"(all {run['energies'].shape[1]} bands: {dft[s][1:, 1].max():.2f} ueV)")
display(pd.DataFrame({"operation": groups["base"][2], "base (ueV)": dft["base"][:, 0].round(3)}))
display(pd.DataFrame({"operation": groups["mode1"][2], **{f"{s} (ueV)": dft[s][:, 0].round(3) for s in ("mode1", "mode2", "mode3")}}))


base  : 47 operations besides the identity: max 1.35 ueV (all 300 bands: 1.35 ueV)
mode1 : 23 operations besides the identity: max 1.34 ueV (all 300 bands: 1.34 ueV)
mode2 : 23 operations besides the identity: max 1.31 ueV (all 300 bands: 1.31 ueV)


mode3 : 23 operations besides the identity: max 1.28 ueV (all 300 bands: 1.28 ueV)


,operation,base (ueV)
0,identity,0.000
1,"180 deg rotation - cart. axis [0,0,1]",0.444
2,"180 deg rotation - cart. axis [0,1,0]",0.444
3,"180 deg rotation - cart. axis [1,0,0]",0.001
4,"180 deg rotation - cart. axis [1,1,0] T",0.503
5,"180 deg rotation - cart. axis [1,-1,0] T",0.503
6,"90 deg rotation - cart. axis [0,0,-1] T",0.503
7,"90 deg rotation - cart. axis [0,0,1] T",0.503
8,"180 deg rotation - cart. axis [1,0,1] T",1.355
9,"180 deg rotation - cart. axis [-1,0,1] T",1.355


,operation,mode1 (ueV),mode2 (ueV),mode3 (ueV)
0,identity,0.000,0.000,0.000
1,"180 deg rotation - cart. axis [0,0,1]",0.498,0.650,0.459
2,"180 deg rotation - cart. axis [0,1,0]",0.498,0.650,0.459
3,"180 deg rotation - cart. axis [1,0,0]",0.001,0.002,0.001
4,"120 deg rotation - cart. axis [-1,-1,-1]",1.339,1.305,1.276
5,"120 deg rotation - cart. axis [-1,1,1]",1.339,1.305,1.276
6,"120 deg rotation - cart. axis [1,1,-1]",1.339,1.305,1.276
7,"120 deg rotation - cart. axis [1,-1,1]",1.339,1.305,1.276
8,"120 deg rotation - cart. axis [1,1,1]",1.339,1.305,1.276
9,"120 deg rotation - cart. axis [-1,1,-1]",1.339,1.305,1.276


## Tests 2 and 3: bands and traced curvature, base

Per operation: the largest band change (meV) and the residual of ω per part. Row 0 is the identity; the controls come
last.

In [6]:
t0 = time.time()
base_folder = RUNS["base"][0]
model = wannier.run_model(wannier90.read_run(base_folder, PREFIX), pin=True, position_matrix=True)   # = pair_models' base
print(f"model in {time.time() - t0:.0f} s")
COLUMNS = ["bands (meV)"] + list(symmetry.PARTS)
results = {}
ops, primed, names, listed = groups["base"]
ctrl = symmetry.controls(ops, primed, CONTROLS)
cases = list(zip(names, ops, primed)) + [("control " + c[0], c[1], c[2]) for c in ctrl]
rows, X0 = symmetry.table(model, k, reciprocal, cases, N_OCC, "vector")
results["base"] = {"names": np.array([c[0] for c in cases]), "n_ops": len(ops), "dft_ueV": dft["base"], "generic": rows}
print("|omega| scale: " + ", ".join(f"{p} {np.abs(X0[p]).max():.3e}" for p in symmetry.PARTS))
display(pd.DataFrame(rows, columns=COLUMNS, index=[c[0] for c in cases])
        .style.format({"bands (meV)": "{:.1e}", **{p: "{:.4%}" for p in symmetry.PARTS}}))


model in 135 s


|omega| scale: internal 6.377e-01, cross 1.323e-02, external 2.403e-02, total 6.471e-01


,bands (meV),internal,cross,external,total
identity,7.1e-11,0.0000%,0.0000%,0.0000%,0.0000%
"180 deg rotation - cart. axis [0,0,1]",2.2e-04,0.0002%,0.0098%,0.0093%,0.0004%
"180 deg rotation - cart. axis [0,1,0]",2.1e-04,0.0002%,0.0094%,0.0063%,0.0002%
"180 deg rotation - cart. axis [1,0,0]",2.4e-04,0.0002%,0.0083%,0.0095%,0.0004%
"180 deg rotation - cart. axis [1,1,0] T",4.3e-04,0.0001%,0.0062%,0.0058%,0.0002%
"180 deg rotation - cart. axis [1,-1,0] T",4.2e-04,0.0002%,0.0067%,0.0039%,0.0002%
"90 deg rotation - cart. axis [0,0,-1] T",4.3e-04,0.0002%,0.0089%,0.0076%,0.0003%
"90 deg rotation - cart. axis [0,0,1] T",4.2e-04,0.0002%,0.0087%,0.0065%,0.0002%
"180 deg rotation - cart. axis [1,0,1] T",5.8e-04,0.0002%,0.0090%,0.0093%,0.0002%
"180 deg rotation - cart. axis [-1,0,1] T",6.3e-04,0.0002%,0.0075%,0.0071%,0.0003%


## Modes 1-3 (cluster)

Their `.chk` are on the cluster only; copy each result to `CLUSTER/<structure>.npz` and it is loaded here.

In [7]:
for s in ("mode1", "mode2", "mode3"):
    f = CLUSTER / f"{s}.npz"
    if f.exists() and "generic" in np.load(f):
        results[s] = dict(np.load(f))
        results[s]["dft_ueV"] = dft[s]
        print(f"{s}: loaded {f}")
    else:
        print(f"{s}: no cluster results yet ({f}); test 1 above")


mode1: no cluster results yet (/Users/treycole/Repos/axion-phonon/data/Y2Ir2O7/symmetry/mode1.npz); test 1 above
mode2: no cluster results yet (/Users/treycole/Repos/axion-phonon/data/Y2Ir2O7/symmetry/mode2.npz); test 1 above
mode3: no cluster results yet (/Users/treycole/Repos/axion-phonon/data/Y2Ir2O7/symmetry/mode3.npz); test 1 above


## Summary: the paper's table

Median and maximum over the operations other than the identity, the controls, and whether every threshold of
`THRESHOLDS` holds. Saved to `SAVE` for the regression tests.

In [8]:
def summarize(label, r):
    n = int(r["n_ops"])
    rows = np.asarray(r["generic"])
    sym, ctrl = rows[1:n], rows[n:]
    row = {"model": label, "operations": n - 1, "DFT max (ueV)": round(float(r["dft_ueV"][1:, 0].max()), 3),
           "bands max (meV)": float(sym[:, 0].max())}
    for i, p in enumerate(symmetry.PARTS):
        row[p] = f"{np.median(sym[:, i + 1]):.4%} ({sym[:, i + 1].max():.4%})"
    names = [str(x) for x in np.asarray(r["names"])[n:]]
    row["controls (total)"] = ", ".join(f"{nm.replace('control ', '')} {v:.0%}" for nm, v in zip(names, ctrl[:, 4]))
    bad = [p for i, p in enumerate(symmetry.PARTS) if sym[:, i + 1].max() > THRESHOLDS[p]]
    bad += ["DFT"] if r["dft_ueV"][1:, 0].max() > THRESHOLDS["dft_ueV"] else []
    bad += ["bands"] if sym[:, 0].max() > THRESHOLDS["bands_meV"] else []
    bad += [f"control {nm}" for nm, v in zip(names, ctrl[:, 4]) if v < THRESHOLDS["control_min"]]
    row["pass"] = "yes" if not bad else "NO: " + ", ".join(bad)
    return row


summary = pd.DataFrame([summarize(label, r) for label, r in results.items()])
display(summary)
if SAVE:
    SAVE.mkdir(exist_ok=True)
    summary.to_csv(SAVE / "summary.csv", index=False)
    for label, r in results.items():
        np.savez(SAVE / f"{label}.npz", **{key: np.asarray(v) for key, v in r.items()})
    print(f"saved {SAVE}")


,model,operations,DFT max (ueV),bands max (meV),internal,cross,external,total,controls (total),pass
0,base,47,1.355,0.000644,0.0002% (0.0003%),0.0083% (0.0114%),0.0065% (0.0107%),0.0003% (0.0004%),"C4z unprimed 200%, S4z unprimed 200%",yes


saved /Users/treycole/Repos/axion-phonon/data/Y2Ir2O7/symmetry
